# Bahan Laporan Tugas Pembelajaran Mendalam

Notebook ini menyiapkan angka, tabel, dan grafik yang dibutuhkan `laporan/LAPORAN.md` tetapi belum ada di keluaran
pipeline. Isinya:

| Bagian | Isi | Untuk laporan |
|---|---|---|
| 1 | Versi Python, library, CPU, RAM, GPU | A.7 |
| 2 | Distribusi tingkat relevansi, jumlah kata, histogram fitur numerik, korelasi antarfitur, contoh ulasan per tingkat | A.3 |
| 3 | Tabel lapisan dan jumlah parameter MLP, jumlah parameter encoder, diagram arsitektur | A.5 |
| 4 | Ringkasan pelatihan setiap run dan kurva loss/NDCG@10 untuk semua seed | A.6c, A.6e |
| 5 | Akurasi pasangan per tingkat relevansi (U, B5, B0), confusion matrix B4, dan uji berpasangan U vs A1–A3 pada data uji | A.5e, A.6e, A.8b |
| 6 | Uji ulang inferensi dari embedding tersimpan, rumus fitur untuk aplikasi Gradio, kecepatan MLP | A.5, A.7d, A.8a |

**Cara menjalankan di Google Colab.** Unggah notebook ini, pilih *Runtime → Change runtime type → T4 GPU* (agar versi
CUDA dan driver ikut tercatat; perhitungannya sendiri cukup dengan CPU), lalu *Run all*. Notebook ini hanya
**membaca** keluaran pipeline di Google Drive dan menulis hasilnya ke folder baru `outputs/<kategori>/laporan/`.
Tidak ada berkas pipeline yang diubah, dan tidak ada model yang dilatih ulang.

**Catatan data uji.** Bagian 5 dan 6 membaca skor data uji yang sudah disimpan oleh notebook 05–06. Analisis ini
hanya merinci hasil yang sudah ada (seperti analisis kesalahan di notebook 07) dan tidak dipakai untuk memilih model
atau hiperparameter.

In [ ]:
#@title Pengaturan
DRIVE_ROOT = "/content/drive/MyDrive/dl-paper"  #@param {type:"string"}
KATEGORI = "Video_Games"  #@param {type:"string"}
SEED = 42

import importlib.util, json, os, platform, subprocess, sys, time
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

# HR_ROOT dan HR_KATEGORI hanya untuk uji lokal (misalnya pada keluaran smoke test)
ROOT = Path(os.environ.get("HR_ROOT", DRIVE_ROOT))
KATEGORI = os.environ.get("HR_KATEGORI", KATEGORI)
OUT = ROOT / "outputs" / KATEGORI
if not (OUT / "config.json").exists():
    raise FileNotFoundError(f"{OUT / 'config.json'} tidak ada. Periksa DRIVE_ROOT dan KATEGORI.")
CFG = json.loads((OUT / "config.json").read_text())
LAP = OUT / "laporan"
FIG, TAB = LAP / "figures", LAP / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)
MS, MT = CFG["main_scale"], CFG["main_target"]
print("keluaran pipeline:", OUT)
print("hasil notebook ini:", LAP)
print("skala utama:", MS, "| target utama:", MT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "serif", "font.size": 10, "figure.dpi": 110,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)
RINGKASAN = {}   # angka kunci untuk LAPORAN.md, disimpan ke laporan/ringkasan.json di akhir


def simpan_gambar(fig, nama):
    p = FIG / f"{nama}.png"
    fig.savefig(p, dpi=300, bbox_inches="tight")
    print("disimpan:", p)
    return p


def tabel_md(df, desimal=4):
    def sel(v):
        if isinstance(v, (float, np.floating)):
            return "" if np.isnan(v) else f"{v:.{desimal}f}"
        return str(v).replace("|", "\\|").replace("\n", " ")
    baris = ["| " + " | ".join(map(str, df.columns)) + " |", "|" + "---|" * len(df.columns)]
    baris += ["| " + " | ".join(sel(v) for v in r) + " |" for r in df.itertuples(index=False)]
    return "\n".join(baris)


def simpan_tabel(df, nama, desimal=4):
    df.to_csv(TAB / f"{nama}.csv", index=False)
    (TAB / f"{nama}.md").write_text(tabel_md(df, desimal) + "\n")
    print("disimpan:", TAB / f"{nama}.csv", "dan .md")
    return df

## 1. Lingkungan (A.7a–c)

Versi dicatat dari runtime tempat notebook ini dijalankan. Run pipeline Video_Games dijalankan di Colab pada
5–6 November 2026 dengan GPU Tesla T4 (tercatat di log notebook `FINISHED RUN`). Colab bisa memperbarui versi
library, jadi jalankan notebook ini secepatnya dan cantumkan tanggalnya di laporan.

In [ ]:
from importlib.metadata import PackageNotFoundError, version

PAKET = ["numpy", "pandas", "pyarrow", "scipy", "scikit-learn", "torch", "transformers", "tokenizers",
         "huggingface-hub", "safetensors", "matplotlib", "orjson", "nbformat", "nbclient", "ipykernel", "gradio"]
versi = []
for p in PAKET:
    try:
        v = version(p)
    except PackageNotFoundError:
        v = "(tidak terpasang di runtime ini)"
    versi.append({"komponen": p, "versi": v})


def nama_cpu():
    try:
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()


import torch
env = {
    "tanggal": time.strftime("%Y-%m-%d %H:%M"),
    "python": platform.python_version(),
    "sistem_operasi": platform.platform(),
    "google_colab": IN_COLAB,
    "cpu": nama_cpu(),
    "jumlah_core_cpu": os.cpu_count(),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1),
    "cuda_tersedia": torch.cuda.is_available(),
    "cuda_versi_torch": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version() if torch.cuda.is_available() else None,
}
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    env["gpu"] = prop.name
    env["gpu_memori_gb"] = round(prop.total_memory / 1e9, 1)
    try:
        env["driver_nvidia"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=20).stdout.strip()
    except (OSError, subprocess.SubprocessError):
        env["driver_nvidia"] = None
(LAP / "lingkungan.json").write_text(json.dumps({"lingkungan": env, "paket": versi}, indent=2))
simpan_tabel(pd.DataFrame(versi), "versi_library")
RINGKASAN["lingkungan"] = env
pd.Series(env)

In [ ]:
pd.DataFrame(versi)

## 2. EDA tambahan (A.3)

Grafik EDA dasar sudah ada di `figures/eda_distribusi.png` (sebaran `helpful_vote`, usia, ulasan per produk, dan
jumlah kata). Bagian ini menambahkan yang diminta format laporan.

Statistik fitur, korelasi, dan contoh ulasan memakai **data latih skala terbesar (E5)**, yaitu baris latih yang
punya target. Dengan begitu tidak ada informasi data validasi atau uji yang dipakai. Distribusi tingkat relevansi
ditampilkan untuk ketiga partisi.

In [ ]:
STRUCT = ["rating", "verified_purchase", "log_review_length", "log_word_count", "log_title_length"]
largest = list(CFG["scales"])[-1]
kolom = ["rid", "parent_asin", "split", f"in_{largest}", "rating", "verified_purchase", "review_length",
         "word_count", "title_length", "helpful_vote", "age_days"] + [f"z_{c}" for c in STRUCT]
df = pd.read_parquet(OUT / "reviews_split.parquet", columns=kolom)
tg = pd.read_parquet(OUT / "targets.parquet", columns=["rid", "y_t1", "grade_t1"])
df = df.merge(tg, on="rid", how="inner")      # hanya baris yang punya target: latih E5, validasi, uji
tr = df[(df["split"] == "train") & df[f"in_{largest}"]]
te = df[df["split"] == "test"]
print(f"baris dengan target: {len(df):,} | latih {largest}: {len(tr):,} | uji: {len(te):,}")

### 2a. Distribusi tingkat relevansi (label)

In [ ]:
NAMA_TINGKAT = {0: "0 (tanpa suara)", 1: "1", 2: "2", 3: "3", 4: "4 (paling membantu)"}
cnt = df.groupby(["split", "grade_t1"]).size().unstack(fill_value=0).reindex(["train", "val", "test"])
prop = cnt.div(cnt.sum(axis=1), axis=0)
t = cnt.T.copy()
t.columns = [f"jumlah_{c}" for c in t.columns]
for c in prop.index:
    t[f"proporsi_{c}"] = prop.loc[c].values
t.insert(0, "tingkat", [NAMA_TINGKAT[g] for g in t.index])
simpan_tabel(t.reset_index(drop=True), "eda_tingkat_relevansi")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
w = 0.27
for k, (s, lab) in enumerate([("train", "latih"), ("val", "validasi"), ("test", "uji")]):
    ax.bar(np.arange(5) + (k - 1) * w, prop.loc[s].values, w, label=lab)
ax.set_xticks(range(5), [NAMA_TINGKAT[g] for g in range(5)], fontsize=8)
ax.set_ylabel("proporsi ulasan")
ax.set_xlabel("tingkat relevansi (dari target T1)")
ax.set_title("Distribusi tingkat relevansi per partisi")
ax.legend()
simpan_gambar(fig, "eda_tingkat_relevansi")
RINGKASAN["proporsi_tingkat_latih"] = prop.loc["train"].round(4).to_dict()
t

### 2b. Jumlah kata (data teks)

In [ ]:
def ringkas_kata(x):
    x = np.asarray(x)
    return {"n": len(x), "rata_rata": x.mean(), "median": np.median(x), "p5": np.quantile(x, .05),
            "p25": np.quantile(x, .25), "p75": np.quantile(x, .75), "p95": np.quantile(x, .95), "maks": x.max()}

rows = [{"kelompok": "semua", **ringkas_kata(tr["word_count"])}]
rows += [{"kelompok": f"tingkat {g}", **ringkas_kata(tr.loc[tr["grade_t1"] == g, "word_count"])} for g in range(5)]
kata = simpan_tabel(pd.DataFrame(rows), "eda_jumlah_kata", desimal=1)

fig, ax = plt.subplots(figsize=(6.5, 3.4))
data = [tr.loc[tr["grade_t1"] == g, "word_count"].clip(lower=1).to_numpy() for g in range(5)]
ax.boxplot(data, showfliers=False)
ax.set_xticks(range(1, 6), [NAMA_TINGKAT[g] for g in range(5)], fontsize=8)
ax.set_yscale("log")
ax.set_ylabel("jumlah kata isi ulasan (skala log)")
ax.set_xlabel("tingkat relevansi")
ax.set_title(f"Jumlah kata per tingkat relevansi (data latih {largest}, tanpa pencilan)")
simpan_gambar(fig, "eda_kata_per_tingkat")

tok = OUT / "tables" / "eda_token.json"
if tok.exists():
    RINGKASAN["token"] = json.loads(tok.read_text())
    print("panjang token (sampel notebook 02):", RINGKASAN["token"])
RINGKASAN["median_kata_per_tingkat"] = kata.set_index("kelompok")["median"].to_dict()
kata

### 2c. Histogram fitur numerik

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 6))
r = tr["rating"].value_counts().sort_index()
axes[0, 0].bar(r.index.astype(int), r.values, color="#4C72B0")
axes[0, 0].set(title="rating", xlabel="bintang", ylabel="jumlah ulasan")
v = tr["verified_purchase"].value_counts().sort_index()
axes[0, 1].bar(["tidak (0)", "ya (1)"], v.reindex([0, 1], fill_value=0).values, color="#4C72B0")
axes[0, 1].set(title="verified_purchase", ylabel="jumlah ulasan")
for ax, c, judul in [(axes[0, 2], "review_length", "panjang isi (karakter)"),
                     (axes[1, 0], "word_count", "jumlah kata isi"),
                     (axes[1, 1], "title_length", "panjang judul (karakter)")]:
    ax.hist(np.log10(tr[c].to_numpy() + 1), bins=50, color="#4C72B0")
    ax.set(title=judul, xlabel=f"log10(1 + {c})", ylabel="frekuensi")
axes[1, 2].hist(np.log1p(tr["helpful_vote"].to_numpy()), bins=50, color="#DD8452")
axes[1, 2].set_yscale("log")
axes[1, 2].set(title="target T1 = ln(1 + helpful_vote)", xlabel="ln(1 + helpful_vote)", ylabel="frekuensi (log)")
fig.suptitle(f"Fitur numerik pada data latih {largest} (oranye = target, bukan fitur masukan)")
fig.tight_layout()
simpan_gambar(fig, "eda_fitur_numerik")

desk = tr[["rating", "verified_purchase", "review_length", "word_count", "title_length", "helpful_vote"]].describe().T
desk.insert(0, "fitur", desk.index)
simpan_tabel(desk.reset_index(drop=True), "eda_statistik_fitur", desimal=3)

### 2d. Korelasi antarfitur

Korelasi Spearman dipakai karena sebaran panjang ulasan dan `helpful_vote` sangat miring. Usia ulasan dan
`helpful_vote` ikut ditampilkan untuk analisis, tetapi **tidak** menjadi fitur masukan model.

In [ ]:
LABEL = {"rating": "rating", "verified_purchase": "verified", "review_length": "panjang isi",
         "word_count": "jumlah kata", "title_length": "panjang judul", "age_days": "usia (hari)*",
         "helpful_vote": "helpful_vote*"}
kor = tr[list(LABEL)].rename(columns=LABEL).corr(method="spearman")
k = kor.copy()
k.insert(0, "fitur", k.index)
simpan_tabel(k.reset_index(drop=True), "eda_korelasi_spearman", desimal=3)

fig, ax = plt.subplots(figsize=(6.4, 5.4))
im = ax.imshow(kor.values, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(kor)), kor.columns, rotation=40, ha="right")
ax.set_yticks(range(len(kor)), kor.index)
for i in range(len(kor)):
    for j in range(len(kor)):
        val = kor.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(val) > 0.6 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, label="korelasi Spearman")
ax.set_title(f"Korelasi Spearman antarfitur (data latih {largest})\n* bukan fitur masukan model")
simpan_gambar(fig, "eda_korelasi")
RINGKASAN["korelasi_spearman_dengan_vote"] = kor["helpful_vote*"].drop("helpful_vote*").round(3).to_dict()
RINGKASAN["korelasi_panjang_isi_vs_jumlah_kata"] = round(float(kor.loc["panjang isi", "jumlah kata"]), 3)
kor.round(3)

### 2e. Contoh ulasan per tingkat relevansi

In [ ]:
import pyarrow.parquet as pq

rng = np.random.default_rng(SEED)
contoh_rid = []
for g in range(5):
    pool = tr.loc[tr["grade_t1"] == g, "rid"].to_numpy()
    contoh_rid += rng.choice(pool, size=min(2, len(pool)), replace=False).tolist()
teks = pq.read_table(OUT / "reviews_split.parquet", columns=["rid", "title", "text"],
                     filters=[("rid", "in", contoh_rid)]).to_pandas()
ex = tr[tr["rid"].isin(contoh_rid)].merge(teks, on="rid")
ex["isi (300 karakter pertama)"] = ex["text"].str.slice(0, 300) + np.where(ex["text"].str.len() > 300, " …", "")
ex = ex.sort_values(["grade_t1", "rid"])
ex = ex[["grade_t1", "helpful_vote", "rating", "verified_purchase", "word_count", "title",
         "isi (300 karakter pertama)"]].rename(columns={"grade_t1": "tingkat", "word_count": "jumlah_kata",
                                                        "title": "judul"})
simpan_tabel(ex.reset_index(drop=True), "eda_contoh_per_tingkat", desimal=0)
ex

## 3. Arsitektur model (A.5b)

Kelas `Scorer` di bawah disalin persis dari `src/hr/models.py`. Bobot model U yang tersimpan dimuat dengan
`strict=True`, jadi tabel lapisan dan jumlah parameter di bawah pasti sama dengan model yang dilatih.

In [ ]:
import torch.nn as nn


class Scorer(nn.Module):
    # Salinan src/hr/models.py
    def __init__(self, d_in: int, hidden=(256, 64), dropout: float = 0.2):
        super().__init__()
        layers, prev = [], d_in
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.float()).squeeze(-1)


KODE = ["B0", "B1", "B2", "B3", "B4", "B5", "U", "A1", "A2", "A3"]     # tanpa run fine-tuning opsional (FT*)
res = pd.DataFrame([json.loads(p.read_text()) for p in sorted((OUT / "runs").glob("*/result.json"))])
res = res[~res["run_id"].str.contains("_tune") & res["code"].isin(KODE)].copy()


def runs(code, scale=MS, target=MT):
    r = res[(res["code"] == code) & (res["scale"] == scale) & (res["target"] == target)]
    return r.sort_values("seed")["run_id"].tolist()


def muat_model(run_id):
    rj = json.loads((OUT / "runs" / run_id / "result.json").read_text())
    sd = torch.load(OUT / "runs" / run_id / "model.pt", map_location="cpu", weights_only=True)
    hidden = tuple(CFG["hidden"]) if rj["model"] == "mlp" else ()
    m = Scorer(sd["net.0.weight"].shape[1], hidden=hidden, dropout=rj["dropout"])
    m.load_state_dict(sd, strict=True)
    return m.eval(), rj


u_run = runs("U")[0]
model_u, rj_u = muat_model(u_run)
d_in = model_u.net[0].in_features
baris = [{"no": 0, "lapisan": f"Masukan x = [e ; z] (embedding {d_in - len(STRUCT)} + {len(STRUCT)} fitur)",
          "ukuran_keluaran": f"(batch, {d_in})", "parameter": 0}]
dim = d_in
for i, layer in enumerate(model_u.net, 1):
    if isinstance(layer, nn.Linear):
        dim, nama = layer.out_features, f"Linear ({layer.in_features} → {layer.out_features})"
    elif isinstance(layer, nn.ReLU):
        nama = "ReLU"
    else:
        nama = f"Dropout (p = {layer.p})"
    baris.append({"no": i, "lapisan": nama, "ukuran_keluaran": f"(batch, {dim})",
                  "parameter": sum(p.numel() for p in layer.parameters())})
lap = pd.DataFrame(baris)
total_u = int(lap["parameter"].sum())
assert total_u == sum(p.numel() for p in model_u.parameters())
lap = pd.concat([lap, pd.DataFrame([{"no": "", "lapisan": "Total (semua dapat dilatih)", "ukuran_keluaran": "skor s (batch,)",
                                     "parameter": total_u}])], ignore_index=True)
simpan_tabel(lap, "arsitektur_U", desimal=0)
RINGKASAN["parameter_mlp_U"] = total_u
print("run:", u_run, "| lr:", rj_u["lr"], "| dropout:", rj_u["dropout"])
lap

In [ ]:
# Jumlah parameter semua model yang dilatih dengan PyTorch (satu seed per kode)
rows = []
for c in ["B2", "B3", "B4", "B5", "U", "A1", "A2", "A3"]:
    r = runs(c)
    if not r:
        continue
    m, rj = muat_model(r[0])
    rows.append({"kode": c, "representasi": rj["rep"], "model": rj["model"], "loss": rj["loss"],
                 "dimensi_masukan": m.net[0].in_features,
                 "lapisan": " → ".join([str(m.net[0].in_features)] +
                                        [str(l.out_features) for l in m.net if isinstance(l, nn.Linear)]),
                 "parameter": sum(p.numel() for p in m.parameters())})
par = simpan_tabel(pd.DataFrame(rows), "parameter_per_model", desimal=0)
par

In [ ]:
# Encoder yang dibekukan. Model dibangun dari konfigurasinya saja (bobot tidak diunduh) untuk menghitung parameter.
# Pipeline memakai last_hidden_state[:, 0] (token <s>/[CLS]), jadi lapisan pooler tidak dipakai.
from transformers import AutoConfig, AutoModel

enc = []
for rep, name in CFG["encoders"].items():
    conf = AutoConfig.from_pretrained(name)
    m = AutoModel.from_config(conf)
    total = sum(p.numel() for p in m.parameters())
    pooler = sum(p.numel() for p in m.pooler.parameters()) if getattr(m, "pooler", None) is not None else 0
    enc.append({"encoder": rep, "checkpoint": name, "lapisan_transformer": conf.num_hidden_layers,
                "dimensi_tersembunyi": conf.hidden_size, "attention_head": conf.num_attention_heads,
                "kosakata": conf.vocab_size, "parameter_total": total, "parameter_dipakai_tanpa_pooler": total - pooler})
    del m
enc = simpan_tabel(pd.DataFrame(enc), "parameter_encoder", desimal=0)
RINGKASAN["parameter_encoder"] = enc.set_index("encoder")["parameter_dipakai_tanpa_pooler"].to_dict()
enc

In [ ]:
# Diagram arsitektur model U
from matplotlib.patches import FancyBboxPatch

enc_par = RINGKASAN["parameter_encoder"].get("blair", 0) / 1e6
n_lap = int(enc.set_index("encoder").loc["blair", "lapisan_transformer"])
d_emb = d_in - len(STRUCT)
hid = list(CFG["hidden"])
kotak = [  # (x, y, teks, warna)
    (0.27, 0.95, "Judul + isi ulasan", "#EEEEEE"),
    (0.27, 0.83, f"Tokenizer BPE RoBERTa\n(pasangan judul–isi, maks {CFG['max_len']} token)", "#EEEEEE"),
    (0.27, 0.69, f"BLaIR-RoBERTa-base (DIBEKUKAN)\n{n_lap} lapisan Transformer, {f"{enc_par:.1f}".replace(".", ",")} juta parameter", "#C9D7EA"),
    (0.27, 0.56, f"Vektor [CLS] + normalisasi L2, e ∈ ℝ^{d_emb}\n(dihitung sekali, disimpan float16)", "#C9D7EA"),
    (0.77, 0.83, "rating, verified_purchase,\nln(1+panjang isi), ln(1+jumlah kata),\nln(1+panjang judul)", "#EEEEEE"),
    (0.77, 0.63, "Standardisasi z-score\n(rata-rata dan simpangan baku data latih)\nz ∈ ℝ^5", "#EEEEEE"),
    (0.50, 0.44, f"Penggabungan x = [e ; z] ∈ ℝ^{d_in}", "#F3E2C7"),
    (0.50, 0.33, f"Linear {d_in}→{hid[0]} + ReLU + Dropout {rj_u['dropout']}", "#D5E8D4"),
    (0.50, 0.23, f"Linear {hid[0]}→{hid[1]} + ReLU + Dropout {rj_u['dropout']}", "#D5E8D4"),
    (0.50, 0.13, f"Linear {hid[1]}→1 (tanpa aktivasi) → skor s", "#D5E8D4"),
    (0.50, 0.02, "Urutkan ulasan dalam satu produk berdasarkan s", "#EEEEEE"),
]
fig, ax = plt.subplots(figsize=(8.5, 9))
ax.set_xlim(0, 1)
ax.set_ylim(-0.04, 1.0)
ax.axis("off")
for x, y, t, c in kotak:
    ax.add_patch(FancyBboxPatch((x - 0.21, y - 0.04), 0.42, 0.08, boxstyle="round,pad=0.005", fc=c, ec="#555555"))
    ax.text(x, y, t, ha="center", va="center", fontsize=8.5)
panah = [((0.27, 0.91), (0.27, 0.87)), ((0.27, 0.79), (0.27, 0.73)), ((0.27, 0.65), (0.27, 0.60)),
         ((0.27, 0.52), (0.45, 0.48)), ((0.77, 0.79), (0.77, 0.67)), ((0.77, 0.59), (0.55, 0.48)),
         ((0.50, 0.40), (0.50, 0.37)), ((0.50, 0.29), (0.50, 0.27)), ((0.50, 0.19), (0.50, 0.17)),
         ((0.50, 0.09), (0.50, 0.06))]
for a, b in panah:
    ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="->", color="#333333"))
ax.text(0.97, 0.23, f"MLP pemberi skor\n{total_u:,} parameter\n(dilatih)".replace(",", "."), ha="right",
        va="center", fontsize=8.5, color="#2E7D32")
ax.set_title("Arsitektur model U. Pelatihan memakai loss RankNet pada pasangan ulasan dari produk yang sama",
             fontsize=9.5)
simpan_gambar(fig, "arsitektur_model")

## 4. Ringkasan pelatihan (A.5c, A.6c, A.6e)

Satu baris per run (tanpa run penalaan). `celah_loss` adalah loss validasi dikurangi loss latih pada epoch
terbaik, sebagai indikasi *overfitting*. `epoch_terbaik` adalah epoch dengan NDCG@10 validasi tertinggi, yang
bobotnya dipakai.

In [ ]:
rows = []
for _, r in res.iterrows():
    h_path = OUT / "runs" / r["run_id"] / "history.csv"
    row = {k: r.get(k) for k in ["run_id", "code", "scale", "target", "seed", "lr", "dropout", "train_reviews",
                                 "epochs_run", "best_epoch", "best_val_ndcg@10", "minutes"]}
    if h_path.exists():
        h = pd.read_csv(h_path)
        b = h.loc[h["val_ndcg@10"].idxmax()]
        row.update(loss_latih_awal=h["train_loss"].iloc[0], loss_latih_terbaik=b["train_loss"],
                   loss_val_terbaik=b["val_ranknet_loss"], celah_loss=b["val_ranknet_loss"] - b["train_loss"],
                   loss_latih_akhir=h["train_loss"].iloc[-1], loss_val_akhir=h["val_ranknet_loss"].iloc[-1],
                   pasangan_per_epoch=h["pairs"].iloc[0], detik_per_epoch=h["seconds"].mean())
    rows.append(row)
lat = pd.DataFrame(rows).sort_values(["code", "scale", "target", "seed"])
simpan_tabel(lat, "ringkasan_pelatihan_per_run")

utama = lat[(lat["scale"] == MS) & (lat["target"] == MT)]
agg = utama.groupby("code").agg(seed=("seed", "count"), epoch_dijalankan=("epochs_run", "mean"),
                                epoch_terbaik=("best_epoch", "mean"), val_ndcg10=("best_val_ndcg@10", "mean"),
                                val_ndcg10_sd=("best_val_ndcg@10", "std"), loss_latih_terbaik=("loss_latih_terbaik", "mean"),
                                loss_val_terbaik=("loss_val_terbaik", "mean"), celah_loss=("celah_loss", "mean"),
                                menit_per_run=("minutes", "mean")).reindex(KODE).dropna(how="all").reset_index()
simpan_tabel(agg, "ringkasan_pelatihan_utama")
RINGKASAN["pelatihan_utama"] = agg.set_index("code").round(4).to_dict(orient="index")
agg

In [ ]:
# Kurva loss dan NDCG@10 validasi untuk semua seed (pengganti grafik akurasi pada tugas pemeringkatan)
kode_kurva = [c for c in ["U", "B5", "B4"] if runs(c)]
fig, axes = plt.subplots(len(kode_kurva), 2, figsize=(10, 3.1 * len(kode_kurva)), squeeze=False)
for i, c in enumerate(kode_kurva):
    for j, run_id in enumerate(runs(c)):
        h = pd.read_csv(OUT / "runs" / run_id / "history.csv")
        seed = run_id.split("_s")[-1]
        warna = f"C{j}"
        axes[i, 0].plot(h["epoch"], h["train_loss"], "-o", ms=3, color=warna, label=f"latih s{seed}")
        axes[i, 0].plot(h["epoch"], h["val_ranknet_loss"], "--s", ms=3, color=warna, label=f"validasi s{seed}")
        axes[i, 1].plot(h["epoch"], h["val_ndcg@10"], "-o", ms=3, color=warna, label=f"s{seed}")
        be = int(h.loc[h["val_ndcg@10"].idxmax(), "epoch"])
        axes[i, 1].axvline(be, color=warna, ls=":", lw=1)
    loss_lbl = "loss (latih = MSE, validasi = RankNet)" if c == "B4" else "loss RankNet"
    axes[i, 0].set(title=f"{c}: loss per epoch", xlabel="epoch", ylabel=loss_lbl)
    axes[i, 1].set(title=f"{c}: NDCG@10 validasi (garis titik = epoch terbaik)", xlabel="epoch", ylabel="NDCG@10")
    axes[i, 0].legend(fontsize=7, ncol=2)
    axes[i, 1].legend(fontsize=7)
fig.tight_layout()
simpan_gambar(fig, "kurva_pelatihan_semua_seed")

## 5. Analisis kesalahan per tingkat relevansi (A.6e, A.8b)

Model pemeringkatan tidak menghasilkan kelas, jadi confusion matrix biasa tidak berlaku untuk U. Penggantinya
adalah **akurasi pasangan per tingkat**. Untuk setiap pasangan ulasan dalam produk yang sama dengan tingkat
a > b, dihitung proporsi pasangan yang skornya berurutan benar (s_a > s_b, skor kembar dihitung setengah).
Nilai 0,5 berarti sama dengan urutan acak. Pasangan dihitung dari seluruh produk data uji dan dirata-ratakan
atas seed.

B4 memprediksi ln(1 + helpful_vote) secara langsung, sehingga prediksinya bisa diubah menjadi tingkat dengan
ambang data latih (`thresholds.json`). Untuk B4 dibuat confusion matrix biasa sebagai pelengkap.

In [ ]:
def muat_skor(run_id, split="test"):
    return pd.read_parquet(OUT / "runs" / run_id / f"pred_{split}.parquet")


def akurasi_pasangan(rows, score):
    # rows: DataFrame dengan parent_asin, grade_t1. score: array sejajar rows.
    benar, total = np.zeros((5, 5)), np.zeros((5, 5))
    pid = rows["parent_asin"].to_numpy()
    g = rows["grade_t1"].to_numpy()
    o = np.argsort(pid, kind="stable")
    pid, g, s = pid[o], g[o], np.asarray(score)[o]
    batas = np.flatnonzero(np.r_[True, pid[1:] != pid[:-1], True])
    for a0, b0 in zip(batas[:-1], batas[1:]):
        gg, ss = g[a0:b0], s[a0:b0]
        if b0 - a0 < 2 or gg.min() == gg.max():
            continue
        for a in range(1, 5):
            sa = ss[gg == a]
            if not len(sa):
                continue
            for b in range(a):
                sb = np.sort(ss[gg == b])
                if not len(sb):
                    continue
                lt = np.searchsorted(sb, sa, side="left")
                le = np.searchsorted(sb, sa, side="right")
                benar[a, b] += lt.sum() + 0.5 * (le - lt).sum()
                total[a, b] += len(sa) * len(sb)
    return benar, total


te_rows = te[["rid", "parent_asin", "grade_t1", "review_length"]].reset_index(drop=True)
hasil = {}
for c in ["U", "B5", "B0"]:
    rr = runs(c)
    if not rr:
        continue
    B, T = np.zeros((5, 5)), np.zeros((5, 5))
    for run_id in rr:
        sc = te_rows[["rid"]].merge(muat_skor(run_id), on="rid", how="left")["score"].to_numpy()
        assert np.isfinite(sc).all(), f"{run_id}: skor uji tidak lengkap"
        b_, t_ = akurasi_pasangan(te_rows, sc)
        B += b_
        T += t_
    hasil[c] = (B, T)

rows = []
for c, (B, T) in hasil.items():
    for a in range(1, 5):
        for b in range(a):
            if T[a, b] > 0:
                rows.append({"model": c, "tingkat_lebih_tinggi": a, "tingkat_lebih_rendah": b,
                             "pasangan": int(T[a, b] / len(runs(c))), "akurasi": B[a, b] / T[a, b]})
    rows.append({"model": c, "tingkat_lebih_tinggi": "semua", "tingkat_lebih_rendah": "semua",
                 "pasangan": int(T.sum() / len(runs(c))), "akurasi": B.sum() / T.sum()})
akp = simpan_tabel(pd.DataFrame(rows), "uji_akurasi_pasangan_per_tingkat")
RINGKASAN["akurasi_pasangan_keseluruhan"] = {c: round(B.sum() / T.sum(), 4) for c, (B, T) in hasil.items()}

fig, axes = plt.subplots(1, len(hasil), figsize=(4.2 * len(hasil), 3.8), squeeze=False)
for ax, (c, (B, T)) in zip(axes[0], hasil.items()):
    M = np.full((4, 4), np.nan)
    for a in range(1, 5):
        for b in range(a):
            if T[a, b] > 0:
                M[a - 1, b] = B[a, b] / T[a, b]
    im = ax.imshow(M, vmin=0.4, vmax=0.9, cmap="viridis")
    for i in range(4):
        for j in range(4):
            if not np.isnan(M[i, j]):
                ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=9,
                        color="white" if M[i, j] < 0.65 else "black")
    ax.set_xticks(range(4), [str(b) for b in range(4)])
    ax.set_yticks(range(4), [str(a) for a in range(1, 5)])
    ax.set(xlabel="tingkat lebih rendah (b)", ylabel="tingkat lebih tinggi (a)",
           title=f"{c} (keseluruhan {B.sum() / T.sum():.3f})")
fig.colorbar(im, ax=axes[0].tolist(), fraction=0.03, label="proporsi pasangan berurutan benar")
fig.suptitle("Akurasi pasangan per tingkat relevansi pada data uji (0,5 = acak)", y=1.02)
simpan_gambar(fig, "uji_akurasi_pasangan_per_tingkat")
akp

In [ ]:
# Confusion matrix B4 (pointwise). Prediksi ln(1+v) diubah menjadi tingkat dengan ambang yang sama dengan label.
q_t1 = np.asarray(json.loads((OUT / "thresholds.json").read_text())["q_t1"])
rr = runs("B4")
if rr:
    CM = np.zeros((5, 5))
    for run_id in rr:
        sc = te_rows[["rid"]].merge(muat_skor(run_id), on="rid", how="left")["score"].to_numpy()
        pred_g = (sc[:, None] > q_t1[None, :]).sum(axis=1)
        np.add.at(CM, (te_rows["grade_t1"].to_numpy(), pred_g), 1)
    CM /= len(rr)
    CMn = CM / CM.sum(axis=1, keepdims=True)
    cm = pd.DataFrame(CMn, columns=[f"prediksi_{j}" for j in range(5)])
    cm.insert(0, "label", [f"tingkat {i}" for i in range(5)])
    cm["jumlah_ulasan"] = CM.sum(axis=1).round().astype(int)
    simpan_tabel(cm, "uji_confusion_matrix_B4", desimal=3)
    RINGKASAN["B4_akurasi_tingkat"] = round(float(np.trace(CM) / CM.sum()), 4)
    RINGKASAN["B4_recall_per_tingkat"] = {i: round(float(CMn[i, i]), 3) for i in range(5)}

    fig, ax = plt.subplots(figsize=(5, 4.3))
    im = ax.imshow(CMn, vmin=0, vmax=1, cmap="Blues")
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{CMn[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if CMn[i, j] > 0.6 else "black")
    ax.set_xticks(range(5), range(5))
    ax.set_yticks(range(5), range(5))
    ax.set(xlabel="tingkat prediksi B4", ylabel="tingkat sebenarnya",
           title=f"Confusion matrix B4 pada data uji\n(dinormalisasi per baris, akurasi {np.trace(CM) / CM.sum():.3f})")
    fig.colorbar(im, ax=ax, fraction=0.046)
    simpan_gambar(fig, "uji_confusion_matrix_B4")
    display(cm)

### 5b. Uji berpasangan ablasi (bukti fitur unik 2)

Notebook 07 hanya menguji selisih berpasangan U terhadap baseline B0–B5. Sel ini menambahkan uji yang sama untuk
U terhadap A1–A3 dengan prosedur yang persis sama (NDCG@10 per produk, rata-rata atas seed, bootstrap per produk,
koreksi Holm). Fungsi di bawah disalin dari `src/hr/metrics.py` dan `src/hr/stats.py`. Sebelum dipakai, hasilnya
dicocokkan dengan NDCG@10 U di `tables/uji_ablasi.csv`.

In [ ]:
def tiebreak_key(rid):
    return (np.asarray(rid, dtype=np.uint64) * np.uint64(2654435761)) % np.uint64(2 ** 32)


def ndcg10_per_produk(rows, score, k=10):
    pid, g, rid = rows["parent_asin"].to_numpy(), rows["grade_t1"].to_numpy(), rows["rid"].to_numpy()
    score = np.asarray(score, dtype=np.float64)
    tb = tiebreak_key(rid)
    srt = np.argsort(pid, kind="stable")
    pid_s = pid[srt]
    bounds = np.flatnonzero(np.r_[True, pid_s[1:] != pid_s[:-1], True])
    disc = 1 / np.log2(np.arange(2, k + 2))
    out = []
    for a, b in zip(bounds[:-1], bounds[1:]):
        idx = srt[a:b]
        if len(idx) < 2:
            continue
        o = idx[np.lexsort((tb[idx], -score[idx]))]
        gp = (2.0 ** g[o][:k].astype(np.float64) - 1)
        gi = (2.0 ** np.sort(g[idx])[::-1][:k].astype(np.float64) - 1)
        idcg = float(np.sum(gi * disc[:len(gi)]))
        out.append((pid_s[a], float(np.sum(gp * disc[:len(gp)])) / idcg if idcg > 0 else np.nan))
    return pd.DataFrame(out, columns=["parent_asin", "ndcg@10"])


def ndcg10_konfigurasi(code):
    pps = []
    for run_id in runs(code):
        sc = te_rows[["rid"]].merge(muat_skor(run_id), on="rid", how="left")["score"].to_numpy()
        pps.append(ndcg10_per_produk(te_rows, sc))
    return pd.concat(pps, ignore_index=True).groupby("parent_asin", as_index=False).mean(numeric_only=True)


def paired_diff(pp_a, pp_b, metric="ndcg@10", n_boot=1000, seed=0):
    m = pp_a[["parent_asin", metric]].merge(pp_b[["parent_asin", metric]], on="parent_asin",
                                            suffixes=("_a", "_b")).dropna()
    d = (m[f"{metric}_a"] - m[f"{metric}_b"]).to_numpy()
    rng = np.random.default_rng(seed)
    boots = d[rng.integers(0, len(d), size=(n_boot, len(d)))].mean(axis=1)
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    return {"diff": float(d.mean()), "lo": float(np.quantile(boots, 0.025)), "hi": float(np.quantile(boots, 0.975)),
            "p": float(min(p, 1.0)), "n": int(len(d)), "win": int((d > 0).sum()), "tie": int((d == 0).sum()),
            "loss": int((d < 0).sum())}


def holm(pvals):
    p = np.asarray(pvals, dtype=np.float64)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(running, 1.0)
    return adj


te_rows = te[["rid", "parent_asin", "grade_t1"]].reset_index(drop=True)
pp = {c: ndcg10_konfigurasi(c) for c in ["U", "A1", "A2", "A3"] if runs(c)}
ref = pd.read_csv(OUT / "tables" / "uji_ablasi.csv")
u_ref = float(ref.loc[ref["code"].str.startswith("A4"), "ndcg@10"].iloc[0])
print(f"NDCG@10 U dihitung ulang {pp['U']['ndcg@10'].mean():.6f} | tersimpan {u_ref:.6f}")
assert abs(pp["U"]["ndcg@10"].mean() - u_ref) < 1e-6, "Perhitungan ulang tidak sama dengan uji_ablasi.csv"

rows = [{"ref": "U", "vs": c, **paired_diff(pp["U"], pp[c], n_boot=CFG["eval_bootstrap"])} for c in ["A1", "A2", "A3"] if c in pp]
abl = pd.DataFrame(rows)
abl["p_holm"] = holm(abl["p"].to_numpy())
abl["lebih_baik"] = (abl["lo"] > 0) & (abl["p_holm"] < 0.05)
simpan_tabel(abl, "uji_selisih_berpasangan_ablasi")
RINGKASAN["uji_berpasangan_ablasi"] = abl.set_index("vs")[["diff", "lo", "hi", "p_holm"]].round(4).to_dict(orient="index")
abl

## 6. Inferensi dan bahan aplikasi Gradio (A.5, A.7d, A.8a)

Sel berikut memeriksa bahwa langkah inferensi yang akan dipakai aplikasi Gradio menghasilkan skor yang sama
dengan skor uji tersimpan:
1. Fitur terstruktur dihitung dari teks mentah dengan rumus pipeline, lalu distandardisasi dengan rata-rata dan
   simpangan baku data latih dari `split_report.json`.
2. Embedding BLaIR dihitung ulang dari judul dan teks dengan encoder yang sama, persis seperti yang akan
   dilakukan aplikasi Gradio. Jika berkas `emb_blair.npy` di Drive utuh, embedding tersimpan juga dipakai untuk
   pemeriksaan ketat kedua.
3. Vektor gabungan diberi skor oleh bobot `model.pt` model U.

Sel pertama memeriksa ukuran berkas embedding di Drive. Berkas berukuran 0 byte atau lebih kecil dari seharusnya
berarti salinan di Drive tidak utuh (lihat catatan pada keluaran sel).

In [ ]:
# Ukuran berkas embedding di Drive dibandingkan ukuran yang seharusnya (baris x dimensi x 2 byte + header).
cek_berkas = []
for rep in ["blair", "roberta", "tfidf_svd"]:
    f, fm = OUT / f"emb_{rep}.npy", OUT / f"emb_{rep}.json"
    if not fm.exists():
        continue
    meta = json.loads(fm.read_text())
    ukuran = f.stat().st_size if f.exists() else -1
    dim = d_emb if rep in ("blair", "roberta") else None
    harap = meta["rows"] * dim * 2 if dim else None
    cek_berkas.append({"berkas": f.name, "ukuran_byte": ukuran,
                       "minimal_seharusnya": harap,
                       "utuh": (ukuran > 0) if harap is None else (ukuran >= harap)})
cek_berkas = pd.DataFrame(cek_berkas)
RINGKASAN["cek_berkas_embedding"] = cek_berkas.to_dict(orient="records")
EMB_UTUH = bool(cek_berkas.loc[cek_berkas["berkas"] == "emb_blair.npy", "utuh"].all())
if not EMB_UTUH:
    print("PERHATIAN: emb_blair.npy di Drive tidak utuh. Hasil run tetap sah karena pelatihan berjalan di runtime "
          "yang sama dengan ekstraksi, tetapi tahap 5/6 tidak bisa dijalankan ulang dari Drive tanpa ekstraksi ulang.\n"
          "Pemeriksaan inferensi di bawah memakai encoder BLaIR langsung.")
cek_berkas

In [ ]:
std = json.loads((OUT / "split_report.json").read_text())["standardizer"]
sample = te.sample(min(2000, len(te)), random_state=SEED)
raw = pd.DataFrame({"rating": sample["rating"].astype(float),
                    "verified_purchase": sample["verified_purchase"].astype(float),
                    "log_review_length": np.log1p(sample["review_length"].astype(float)),
                    "log_word_count": np.log1p(sample["word_count"].astype(float)),
                    "log_title_length": np.log1p(sample["title_length"].astype(float))})
z_manual = np.column_stack([(raw[c] - std[c][0]) / std[c][1] for c in STRUCT]).astype(np.float32)
z_simpan = sample[[f"z_{c}" for c in STRUCT]].to_numpy(np.float32)
selisih_z = float(np.abs(z_manual - z_simpan).max())
print("selisih maksimum z (rumus vs tersimpan):", selisih_z)
assert selisih_z < 1e-3

tersimpan = sample[["rid"]].merge(muat_skor(u_run), on="rid", how="left")["score"].to_numpy()
assert not np.isnan(tersimpan).any()

def skor_u(e, z):
    x = np.hstack([e.astype(np.float16), z.astype(np.float16)])   # sama dengan features.fuse
    with torch.no_grad():
        return model_u(torch.as_tensor(x)).numpy()

# (a) Encoder BLaIR dijalankan langsung, sama dengan embed.extract_embeddings dan aplikasi Gradio.
import torch.nn.functional as F
from scipy.stats import spearmanr
from transformers import AutoModel, AutoTokenizer
eb = json.loads((OUT / "emb_blair.json").read_text())
dev = "cuda" if torch.cuda.is_available() else "cpu"
n_enc = len(sample) if dev == "cuda" else min(300, len(sample))      # CPU lambat, cukup 300 ulasan
teks = (pq.read_table(OUT / "reviews_split.parquet", columns=["rid", "title", "text"],
                      filters=[("rid", "in", sample["rid"].iloc[:n_enc].tolist())])
        .to_pandas().set_index("rid").loc[sample["rid"].iloc[:n_enc]])
tok = AutoTokenizer.from_pretrained(eb["model"])
enc_model = AutoModel.from_pretrained(eb["model"]).to(dev).eval()
if dev == "cuda":
    enc_model = enc_model.half()                                     # sama dengan run (fp16 di GPU)
judul = [" ".join((t or "").split()[:64]) for t in teks["title"]]    # embed._title_for_pair
isi = teks["text"].tolist()
e_enc = np.empty((n_enc, d_emb), dtype=np.float16)
urut_pjg = np.argsort([len(t) for t in isi], kind="stable")
t0 = time.time()
with torch.inference_mode():
    for i in range(0, n_enc, 64):
        b = urut_pjg[i:i + 64]
        enc = tok([judul[k] for k in b], [isi[k] for k in b], truncation="only_second",
                  max_length=eb["max_len"], padding=True, return_tensors="pt").to(dev)
        h = F.normalize(enc_model(**enc).last_hidden_state[:, 0].float(), dim=-1)
        e_enc[b] = h.cpu().numpy().astype(np.float16)
detik_enc = time.time() - t0
s_enc = skor_u(e_enc, z_manual[:n_enc])
rho_enc = float(spearmanr(s_enc, tersimpan[:n_enc]).statistic)
selisih_enc = float(np.abs(s_enc - tersimpan[:n_enc]).max())
print(f"(a) encoder langsung di {dev}, {n_enc} ulasan, {n_enc / detik_enc:.0f} ulasan/detik")
print(f"    selisih maksimum skor = {selisih_enc:.2e}, Spearman = {rho_enc:.6f}, "
      f"simpangan baku skor tersimpan = {tersimpan.std():.3f}")
assert rho_enc > 0.99, "Skor dari encoder langsung tidak sesuai dengan skor tersimpan"
del enc_model
if dev == "cuda":
    torch.cuda.empty_cache()

# (b) Embedding tersimpan di Drive, hanya jika berkasnya utuh. Seharusnya identik (selisih ~1e-8).
selisih_s = None
if EMB_UTUH:
    idx = pd.read_parquet(OUT / "emb_index.parquet")
    pos = pd.Series(np.arange(len(idx)), index=idx["rid"].to_numpy())
    E = np.load(OUT / "emb_blair.npy", mmap_mode="r")
    p = pos.loc[sample["rid"]].to_numpy()
    urut = np.argsort(p)
    e = np.empty((len(p), E.shape[1]), dtype=np.float16)
    e[urut] = np.asarray(E[p[urut]])
    s = skor_u(e, z_manual)
    selisih_s = float(np.abs(s - tersimpan).max())
    print(f"(b) embedding tersimpan: selisih maksimum skor U ({u_run}) dibanding pred_test.parquet:", selisih_s)
    assert selisih_s < 1e-2, "Inferensi ulang tidak sama dengan skor tersimpan"
else:
    print("(b) dilewati karena emb_blair.npy di Drive tidak utuh")
RINGKASAN["uji_inferensi_ulang"] = {"ulasan": int(len(sample)), "selisih_maks_z": selisih_z,
                                     "encoder_langsung": {"perangkat": dev, "ulasan": int(n_enc),
                                                          "selisih_maks_skor": selisih_enc, "spearman": rho_enc,
                                                          "ulasan_per_detik": round(n_enc / detik_enc, 1)},
                                     "embedding_tersimpan_selisih_maks_skor": selisih_s}
RINGKASAN["standardisasi"] = std
pd.DataFrame({"fitur": STRUCT, "rata_rata_latih": [std[c][0] for c in STRUCT],
              "simpangan_baku_latih": [std[c][1] for c in STRUCT]})

In [ ]:
# Kecepatan MLP U (tanpa encoder). Encoder adalah bagian paling lambat, lihat emb_blair.json.
speed = {}
for dev in ["cpu"] + (["cuda"] if torch.cuda.is_available() else []):
    m = model_u.to(dev)
    xb = torch.randn(100_000, d_in, device=dev)
    with torch.no_grad():
        m(xb[:1000])
        if dev == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        m(xb)
        if dev == "cuda":
            torch.cuda.synchronize()
    speed[dev] = round(100_000 / (time.time() - t0))
model_u.to("cpu")
eb = json.loads((OUT / "emb_blair.json").read_text())
speed["encoder_blair_ulasan_per_detik_saat_run"] = round(eb["done"] / eb["elapsed_s"], 1)
speed["encoder_blair_jam_saat_run"] = round(eb["elapsed_s"] / 3600, 2)
RINGKASAN["kecepatan_ulasan_per_detik"] = speed
pd.Series(speed)

In [ ]:
(LAP / "ringkasan.json").write_text(json.dumps(RINGKASAN, indent=2, default=str))
print("ringkasan:", LAP / "ringkasan.json")
print("\nberkas yang dibuat:")
for f in sorted(LAP.rglob("*")):
    if f.is_file():
        print(" ", f.relative_to(LAP))

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    print("Drive di-flush dan di-unmount. Semua hasil sudah tersimpan.")